# CrimeBase quickstart: India crime by state and year

Official NCRB statistics, cleaned into one table where every number points back to the document it came from. This notebook loads it, shows what is covered, draws three charts and follows one number back to its source.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

D = '/kaggle/input/crimebase' if os.path.exists('/kaggle/input/crimebase') else os.environ.get('CRIMEBASE_DIR', '.')
p = pd.read_csv(f'{D}/crime_state_year.csv')
print(f'{len(p):,} rows | {p.state_code.nunique()} states and UTs | {int(p.year.min())} to {int(p.year.max())}')
p.head()

## What is in it
One row per **state / UT x year x crime head x measure**. `scope` is the NCRB table family the number comes from (`ipc`, `sll`, `women`, `children`, ...). The same head appears in several families that overlap, so always filter on `scope`; never add across scopes. Only numbers whose state rows add up to NCRB's own all-India row are included, and `disputed` marks the few where files disagree.

In [ ]:
print(p.groupby(['scope', 'metric']).size().unstack(fill_value=0))
print()
print(p.groupby(['year', 'scope']).size().unstack(fill_value=0))

## Murder cases registered in India under the IPC

In [ ]:
n = pd.read_csv(f'{D}/crime_india_year.csv')   # all-India series: national tables plus the all-India row of each state table
m = n[(n.scope == 'ipc') & (n.crime_head == 'Murder') & (n.metric == 'cases_registered')].sort_values('year')
ax = m.plot(x='year', y='value', marker='o', figsize=(8, 4), title='Murder cases registered under the IPC', legend=False)
ax.set_ylabel('cases'); ax.set_xlabel('')
plt.show()

## Which states reported the most in the latest year available?

In [ ]:
sm = p[(p.scope == 'ipc') & (p.crime_head == 'Murder') & (p.metric == 'cases_registered')]
y = int(sm.year.max())
top = sm[sm.year == y].nlargest(10, 'value')[['state', 'value']].set_index('state')
ax = top.value.sort_values().plot.barh(figsize=(8, 4), title=f'Murder cases registered, top 10 states, {y}')
ax.set_ylabel('')
plt.show()

## Crimes against women: dowry deaths

In [ ]:
w = p[(p.scope == 'women') & (p.crime_head == 'Dowry Deaths') & (p.metric == 'cases_registered')]
ax = w.groupby('year').value.sum().plot(marker='o', figsize=(8, 4), title='Dowry deaths registered (crimes against women table)')
ax.set_ylabel('cases'); ax.set_xlabel('')
plt.show()

## Follow a number back to its source
`file` joins to `documents.csv`, which holds the original URL and the Internet Archive permalink of the exact capture. The document itself is in the companion dataset `crimebase-raw` as `raw-<first character of file>/<file>`.

In [ ]:
docs = pd.read_csv(f'{D}/documents.csv')
r = sm.merge(docs[['file', 'title_hint', 'source_url', 'wayback_url']], on='file').head(1)
r[['state', 'year', 'value', 'title_hint', 'source_url', 'wayback_url']].T

## Crime news

In [ ]:
e = pd.read_parquet(f'{D}/news_events.parquet')
print(f'{len(e):,} distinct events from {e.n_publishers.sum():,} outlet reports')
ax = e.groupby('category').size().sort_values().plot.barh(figsize=(8, 5), title='Crime news events by category')
ax.set_ylabel('')
plt.show()

## Caveats
Reported crime is not actual crime: registration practices differ by state and year. Coverage follows what NCRB published as spreadsheets (see `PROFILE.md` in the dataset). Details of every column: `DATA_DICTIONARY.md`.

Code and method: https://github.com/chandranshB/CrimeBase. If this helps your work, a citation is appreciated.